In [28]:
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.utils.data as data
import wandb
import os

In [29]:
!pip install lightly

In [30]:
from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform

In [31]:
if os.path.exists("/kaggle/working"):
    base_path = "/kaggle/working"

elif "google.colab" in str(get_ipython()):
    base_path = "/content/drive/MyDrive/ProjectSimCLR"

else:
    base_path = "./"


def save_checkpoint(model, optimizer, epoch, loss, run_id):

    torch.save(
        {
            "epoch": epoch,
            "optimizer": optimizer,
            "model": model,
            "loss": loss,
            "wandb_run_id": run_id,
        },
        os.path.join(base_path, f"{run_id}_epoch_{epoch}"),
    )


def load_checkpoint(model, optimizer, path, map_location=None):
    """Loads checkpoints produced by save_checkpoint (pickled model and optimizer objects)."""
    if map_location is None:
        if torch.cuda.is_available():
            map_location = "cuda"
        elif torch.backends.mps.is_available():
            map_location = "mps"
        else:
            map_location = "cpu"
    checkpoint = torch.load(path, weights_only=False, map_location=map_location)
    saved_model = checkpoint["model"]
    saved_optimizer = checkpoint["optimizer"]
    model.load_state_dict(saved_model.state_dict())
    optimizer.load_state_dict(saved_optimizer.state_dict())
    return checkpoint["epoch"], checkpoint["loss"], checkpoint["wandb_run_id"]

In [32]:
class SimCLR(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, 1)
        x = self.projection_head(x)
        return x


In [33]:
resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(input_dim=512, hidden_dim=512, output_dim=128)
model = SimCLR(backbone, projection_head)

In [34]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)

SimCLR(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [ ]:
# Uncomment for running on kaggle
"""
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_KEY")
wandb.login(key=wandb_key)
"""

'\nfrom kaggle_secrets import UserSecretsClient\nuser_secrets = UserSecretsClient()\nwandb_key = user_secrets.get_secret("WANDB_KEY")\nwandb.login(key=wandb_key)\n'

In [ ]:
hp = {
    "batch_size": 256,
    "pipeline_test": True,
    "epochs": 50,
    "temperature": 0.2, 
    "optim": {"base_lr": 0.06, "weight_decay": 1e-5},
}

In [37]:
transform = SimCLRTransform(input_size=32, gaussian_blur=0.0)
dataset = datasets.CIFAR10(
    root="./data", train=True, transform=transform, download=True
)
if hp["pipeline_test"]:
    dataset = data.Subset(dataset, range(256))

/Users/hannesneumann/Repositories/SelfSupervisedLearning/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [38]:
dataloader = DataLoader(
    dataset, batch_size=hp["batch_size"], shuffle=True, drop_last=True, num_workers=2
)

In [46]:
criterion = NTXentLoss(temperature=hp["temperature"])
optimizer = torch.optim.SGD(params=model.named_parameters(), lr=hp["optim"]["base_lr"] * hp["batch_size"] / 256, weight_decay=hp["optim"]["weight_decay"])
# Uncomment when using multiple GPUs
model = torch.nn.DataParallel(model)

In [ ]:
print("Starting Training ...")
run = wandb.init(project="simclr-cifar10", config=hp, mode='online')
model = model.to(device)
for epoch in range(hp["epochs"]):
    print(f"Epoch {epoch} of {hp['epochs']}")
    total_loss = 0
    for batch in dataloader:
        x0, x1 = batch[0]
        x0 = x0.to(device)
        x1 = x1.to(device)
        z0 = model(x0)
        z1 = model(x1)

        loss = criterion(z0, z1)
        total_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    avg_loss = total_loss / len(dataloader)
    wandb.log({"loss": avg_loss, "epoch": epoch, "lr": optimizer.param_groups[0]["lr"]})
    if epoch % 10 == 0 and epoch >= 20:
        save_checkpoint(model, optimizer, epoch, loss, run.id)
    print(f"Epoch: {epoch}, Avg_loss: {avg_loss:.5f}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/hannesneumann/.netrc.


Starting Training ...


wandb: Currently logged in as: hneumann (hneumann-university-of-mannheim) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0 of 50
Epoch: 0, Avg_loss: 5.56313
Epoch 1 of 50
Epoch: 1, Avg_loss: 5.52196
Epoch 2 of 50
Epoch: 2, Avg_loss: 5.50286
Epoch 3 of 50
Epoch: 3, Avg_loss: 5.54611
Epoch 4 of 50
Epoch: 4, Avg_loss: 5.52359
Epoch 5 of 50
Epoch: 5, Avg_loss: 5.54764
Epoch 6 of 50
Epoch: 6, Avg_loss: 5.49798
Epoch 7 of 50
Epoch: 7, Avg_loss: 5.57051
Epoch 8 of 50
Epoch: 8, Avg_loss: 5.44012
Epoch 9 of 50
Epoch: 9, Avg_loss: 5.48918
Epoch 10 of 50
Epoch: 10, Avg_loss: 5.52751
Epoch 11 of 50
Epoch: 11, Avg_loss: 5.41152
Epoch 12 of 50
Epoch: 12, Avg_loss: 5.45578
Epoch 13 of 50
Epoch: 13, Avg_loss: 5.43608
Epoch 14 of 50
Epoch: 14, Avg_loss: 5.31312
Epoch 15 of 50
Epoch: 15, Avg_loss: 5.35868
Epoch 16 of 50
Epoch: 16, Avg_loss: 5.35934
Epoch 17 of 50
Epoch: 17, Avg_loss: 5.55204
Epoch 18 of 50
Epoch: 18, Avg_loss: 5.36055
Epoch 19 of 50
Epoch: 19, Avg_loss: 5.27658
Epoch 20 of 50
Epoch: 20, Avg_loss: 5.36983
Epoch 21 of 50
Epoch: 21, Avg_loss: 5.45798
Epoch 22 of 50
Epoch: 22, Avg_loss: 5.27932
Epoch 23

In [41]:
if torch.cuda.is_available():
    torch_device = torch.device("cuda")
elif torch.backends.mps.is_available():
    torch_device = torch.device("mps")
else:
    torch_device = torch.device("cpu")

resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(input_dim=512, hidden_dim=512, output_dim=128)
model = SimCLR(backbone, projection_head)
optimizer = torch.optim.SGD(params=model.named_parameters(), **hp["optim"])
epoch, loss, run_id = load_checkpoint(
    model,
    optimizer,
    "/Users/hannesneumann/Repositories/SelfSupervisedLearning/models/kxv5e225_epoch_20.pth",
    map_location=torch_device,
)
model.to(torch_device)
print(model)

SimCLR(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [42]:
def linear_evaluation(backbone, train_loader, test_loader, device):
    # Encoder einfrieren
    for param in backbone.parameters():
        param.requires_grad = False

    classifier = nn.Linear(512, 10).to(device)  # 10 CIFAR-10 Klassen
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    # Nur Classifier trainieren
    total_loss = 0
    for epoch in range(20):
        for imgs, labels in train_loader:
            features = backbone(imgs.to(device))
            preds = classifier(features)
            loss = nn.CrossEntropyLoss()(preds, labels.to(device))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss
        
    
    # Evaluate the models performance on the test set
    model.eval()
    classifier.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            features = backbone(imgs.to(device))
            preds = classifier(features)
            predicted = preds.argmax(dim=1)
            correct += (predicted == labels.to(device)).sum().item()
            total += labels.size(0)
    accuracy = correct / total if total > 0 else 0


    wandb.log({"linear_eval_accuracy": accuracy})

In [43]:
test_set =  datasets.CIFAR10(root="./data", train=False, download=True)
test_looader = DataLoader(test_set)
linear_evaluation(model.encoder, dataloader, test_looader, device)


/Users/hannesneumann/Repositories/SelfSupervisedLearning/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


AttributeError: 'list' object has no attribute 'to'